# Simulación de datos — Sistema inteligente de cobranzas
### Proyecto Integrador de Inteligencia Artificial

Versión corregida con reglas de negocio actualizadas:
- sin `limite_credito`
- `estado_pago` renombrado a `target_mora`
- facturas ordenadas cronológicamente por `factura_id`
- gestiones preventivas y reactivas con catálogos coherentes
- promesas sin `monto_comprometido`, sin `fecha_cumplimiento_real` y sin `dias_retraso_promesa`
- features construidas por `fecha_corte` sin fuga temporal deliberada

## 1. Instalación de dependencias

In [1]:
# Dependencias ya cubiertas en este entorno.
print('Dependencias listas.')

Dependencias listas.


## 2. Importaciones y configuración de semilla

In [2]:
import random
import sys
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'project_paths.py').exists():
            return candidate
    raise FileNotFoundError('No se pudo localizar la raiz del proyecto.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from project_paths import GENERATION_DIR, ensure_artifact_dirs

ensure_artifact_dirs()
OUTPUT_DIR = GENERATION_DIR

# Semilla fija para reproducibilidad
# Cambiar este valor genera un dataset diferente pero igualmente válido
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('es_MX')
Faker.seed(SEED)

print('Librerías cargadas correctamente.')
print(f'Carpeta canónica de salida: {OUTPUT_DIR}')

Librerías cargadas correctamente.
Carpeta canónica de salida: C:\Users\KevinUbe\Documents\Proyectos\GRUPO_3\GRUPO_3\artifacts\01_generacion


## 3. Parámetros globales de la simulación

Estos valores controlan el tamaño y comportamiento general del dataset.
Puedes ajustarlos sin romper la lógica del sistema.

In [3]:
# ── Dimensiones del dataset ───────────────────────────────────────────────────
N_CLIENTES      = 200
FECHA_INICIO    = datetime(2023, 1, 1)
FECHA_FIN       = datetime(2024, 12, 31)

# ── Catálogos ─────────────────────────────────────────────────────────────────
SECTORES = [
    'retail', 'manufactura', 'servicios', 'construccion',
    'agro', 'tecnologia', 'salud', 'transporte'
]

# Plazos de crédito posibles en días
CONDICIONES_CREDITO = [30, 45, 60, 90]

# Clases de mora (target del modelo)
CLASES_MORA = ['on_time', '+30', '+60', '+90']

# Rango de días reales de mora por categoría
MORA_DIAS = {
    'on_time': (0,   0),
    '+30':     (1,  30),
    '+60':     (31, 60),
    '+90':     (61, 120),
}

# Escalamiento de canales según número de intento (0-indexado)
CANALES_ESCALAMIENTO = [
    'whatsapp',
    'email',
    'llamada',
    'llamada',
    'visita',
    'visita',
    'carta_notarial',
]

# Probabilidad base de contacto/entrega por canal
PROB_CONTACTO = {
    'whatsapp':       0.65,
    'email':          0.40,
    'llamada':        0.55,
    'visita':         0.70,
    'carta_notarial': 0.90,
}

# Encoding numérico de resultado para features
RESULTADO_ENC = {
    'pagado':             0,
    'confirma_pago':      1,
    'promesa_de_pago':    2,
    'en_proceso_interno': 3,
    'disputa_monto':      4,
    'cliente_ausente':    5,
    'no_contesta':        6,
    'numero_invalido':    7,
    'rechazo_pago':       8,
}

RESULTADOS_NO_CONTACTO = {'no_contesta', 'numero_invalido', 'cliente_ausente'}
RESULTADOS_CONTACTO    = {
    'pagado', 'confirma_pago', 'promesa_de_pago',
    'en_proceso_interno', 'disputa_monto', 'rechazo_pago'
}

# Número de gestiones esperadas por clase de mora
GESTIONES_POR_CLASE = {
    'on_time': (0, 1),
    '+30':     (1, 3),
    '+60':     (3, 6),
    '+90':     (5, 10),
}

# Motivos de no pago posibles
MOTIVOS_NO_PAGO = [
    'flujo_de_caja',
    'disputa_factura',
    'problemas_internos',
    'desconoce_deuda',
    'en_proceso_de_pago',
    'promete_pagar',
    'rechazo_total',
]

print('Parámetros configurados.')

Parámetros configurados.


## 4. Perfiles de pago

Cada cliente tiene un perfil interno que determina cómo se comporta.
Este perfil **no entra al modelo** — el modelo lo debe inferir desde el historial.
Solo sirve para controlar la simulación.

In [4]:
PERFILES = {
    'excelente': {
        'prob':         0.25,
        'pago':         [0.85, 0.10, 0.04, 0.01],
        'prob_promesa': 0.84,
    },
    'regular': {
        'prob':         0.40,
        'pago':         [0.55, 0.25, 0.15, 0.05],
        'prob_promesa': 0.59,
    },
    'riesgoso': {
        'prob':         0.25,
        'pago':         [0.20, 0.30, 0.30, 0.20],
        'prob_promesa': 0.30,
    },
    'critico': {
        'prob':         0.10,
        'pago':         [0.05, 0.15, 0.30, 0.50],
        'prob_promesa': 0.14,
    },
}

# Resultados cuando SÍ hubo contacto efectivo, según el momento de la gestión
RESULTADOS_PREVENTIVOS = {
    'excelente': {
        'confirma_pago':      0.65,
        'en_proceso_interno': 0.20,
        'promesa_de_pago':    0.10,
        'disputa_monto':      0.05,
    },
    'regular': {
        'confirma_pago':      0.45,
        'en_proceso_interno': 0.25,
        'promesa_de_pago':    0.20,
        'disputa_monto':      0.10,
    },
    'riesgoso': {
        'confirma_pago':      0.20,
        'en_proceso_interno': 0.30,
        'promesa_de_pago':    0.35,
        'disputa_monto':      0.15,
    },
    'critico': {
        'confirma_pago':      0.10,
        'en_proceso_interno': 0.25,
        'promesa_de_pago':    0.40,
        'disputa_monto':      0.25,
    },
}

RESULTADOS_REACTIVOS_POR_CLASE = {
    '+30': {
        'promesa_de_pago':    0.40,
        'en_proceso_interno': 0.20,
        'pagado':             0.15,
        'disputa_monto':      0.10,
        'confirma_pago':      0.10,
        'rechazo_pago':       0.05,
    },
    '+60': {
        'promesa_de_pago':    0.28,
        'disputa_monto':      0.18,
        'rechazo_pago':       0.18,
        'en_proceso_interno': 0.16,
        'pagado':             0.12,
        'confirma_pago':      0.08,
    },
    '+90': {
        'rechazo_pago':       0.28,
        'promesa_de_pago':    0.20,
        'disputa_monto':      0.18,
        'en_proceso_interno': 0.16,
        'pagado':             0.10,
        'confirma_pago':      0.08,
    },
}

print('Perfiles de pago configurados.')

Perfiles de pago configurados.


## 5. Funciones auxiliares

Funciones reutilizables que encapsulan la lógica de simulación.

In [5]:
def elegir_perfil() -> str:
    """Selecciona un perfil de pago según las probabilidades definidas."""
    perfiles = list(PERFILES.keys())
    probs    = [PERFILES[p]['prob'] for p in perfiles]
    return np.random.choice(perfiles, p=probs)


def simular_clase_mora(perfil: str, moras_consecutivas: int) -> str:
    """
    Determina la clase de mora de una factura.
    Si el cliente acumula moras consecutivas, sus probabilidades se deterioran.
    """
    probs = PERFILES[perfil]['pago'].copy()

    if moras_consecutivas > 0:
        factor   = min(moras_consecutivas * 0.05, 0.30)
        probs[0] = max(probs[0] - factor, 0.02)
        exceso   = factor / 3
        probs[1] = min(probs[1] + exceso, 0.60)
        probs[2] = min(probs[2] + exceso, 0.60)
        probs[3] = min(probs[3] + exceso, 0.80)
        total    = sum(probs)
        probs    = [p / total for p in probs]

    return np.random.choice(CLASES_MORA, p=probs)


def dias_mora_reales(clase: str) -> int:
    """Genera un número real de días de mora dentro del rango de cada clase."""
    lo, hi = MORA_DIAS[clase]
    return 0 if lo == hi else int(np.random.randint(lo, hi + 1))


def monto_factura(sector: str) -> float:
    """Genera un monto realista según el sector económico del cliente."""
    rangos = {
        'retail':       (500,    15_000),
        'manufactura':  (2_000,  80_000),
        'servicios':    (300,    10_000),
        'construccion': (5_000, 150_000),
        'agro':         (1_000,  40_000),
        'tecnologia':   (800,    25_000),
        'salud':        (400,    12_000),
        'transporte':   (1_500,  30_000),
    }
    lo, hi = rangos[sector]
    return round(random.uniform(lo, hi), 2)


def fechas_en_rango(inicio: datetime, fin: datetime, n: int) -> list:
    """Genera n fechas de emisión distribuidas aleatoriamente en el rango dado."""
    delta = (fin - inicio).days
    if n <= 0 or delta <= 0:
        return []
    dias_ordenados = sorted(random.sample(range(delta + 1), min(n, delta + 1)))
    return [inicio + timedelta(days=d) for d in dias_ordenados]


def muestrear_fechas_unicas(inicio: datetime, fin: datetime, n: int) -> list:
    """Devuelve n fechas únicas y ordenadas dentro de un rango inclusivo."""
    if n <= 0 or inicio > fin:
        return []
    total_dias = (fin - inicio).days
    if total_dias + 1 < n:
        # Si la ventana no alcanza, se usa todo el rango disponible.
        n = total_dias + 1
    offsets = sorted(random.sample(range(total_dias + 1), n))
    return [inicio + timedelta(days=o) for o in offsets]


def canal_para_intento(numero_intento: int) -> str:
    """Devuelve el canal a usar según el número de intento (escalamiento)."""
    idx = min(numero_intento, len(CANALES_ESCALAMIENTO) - 1)
    return CANALES_ESCALAMIENTO[idx]


def resultado_no_contacto(canal: str) -> str:
    """Asigna un resultado coherente cuando no hubo contacto efectivo."""
    if canal == 'visita':
        return 'cliente_ausente'
    if canal == 'carta_notarial':
        return 'cliente_ausente'
    if canal in ('whatsapp', 'llamada'):
        return np.random.choice(['no_contesta', 'numero_invalido'], p=[0.88, 0.12])
    return 'no_contesta'


def simular_resultado_gestion(etapa: str, clase_mora: str, perfil: str, canal: str, numero_intento: int) -> str:
    """
    Determina el resultado de una gestión exitosa.
    - Preventiva: prima la confirmación o señales tempranas.
    - Reactiva: depende del tramo final de mora.
    """
    if etapa == 'preventiva':
        base = RESULTADOS_PREVENTIVOS[perfil].copy()
    else:
        base = RESULTADOS_REACTIVOS_POR_CLASE.get(clase_mora, RESULTADOS_REACTIVOS_POR_CLASE['+30']).copy()

        # Perfiles más riesgosos: menos pago, más rechazo o disputa
        if perfil in ('riesgoso', 'critico'):
            base['rechazo_pago']       = base.get('rechazo_pago', 0) + 0.06
            base['disputa_monto']      = base.get('disputa_monto', 0) + 0.03
            base['pagado']             = max(base.get('pagado', 0) - 0.04, 0.02)
            base['confirma_pago']      = max(base.get('confirma_pago', 0) - 0.03, 0.01)

        if numero_intento >= 4:
            base['rechazo_pago']       = base.get('rechazo_pago', 0) + 0.05
            base['promesa_de_pago']    = max(base.get('promesa_de_pago', 0) - 0.03, 0.02)

        if canal == 'carta_notarial':
            base['rechazo_pago']       = base.get('rechazo_pago', 0) + 0.08
            base['promesa_de_pago']    = max(base.get('promesa_de_pago', 0) - 0.05, 0.02)

    resultados = list(base.keys())
    probs      = np.array(list(base.values()), dtype=float)
    probs      = probs / probs.sum()
    return np.random.choice(resultados, p=probs)


def fecha_compromiso_desde_pago(fecha_promesa: pd.Timestamp, fecha_pago_real: pd.Timestamp, perfil: str) -> pd.Timestamp:
    """
    Construye la fecha compromiso de manera que el cumplimiento observado
    se derive de la relación entre la fecha real de pago y la fecha prometida.
    """
    fecha_promesa  = pd.to_datetime(fecha_promesa)
    fecha_pago_real = pd.to_datetime(fecha_pago_real)
    earliest       = fecha_promesa + timedelta(days=1)
    prob_cumple    = PERFILES[perfil]['prob_promesa']

    # Se induce una probabilidad aproximada de cumplimiento,
    # pero el campo final se calcula únicamente comparando fechas.
    if earliest >= fecha_pago_real:
        return earliest

    if random.random() < prob_cumple:
        lo = max(earliest, fecha_pago_real)
        hi = lo + timedelta(days=random.randint(0, 10))
        return lo + timedelta(days=random.randint(0, (hi - lo).days))
    else:
        hi = fecha_pago_real - timedelta(days=1)
        if earliest > hi:
            return fecha_pago_real
        return earliest + timedelta(days=random.randint(0, (hi - earliest).days))


def promesa_activa_en_corte(promesas_factura: pd.DataFrame, fecha_corte: pd.Timestamp, fecha_pago_real: pd.Timestamp) -> int:
    """
    Una promesa está activa si:
    - ya fue creada al corte
    - la última promesa vigente aún no vence
    - la factura aún no ha sido pagada al corte
    """
    if len(promesas_factura) == 0:
        return 0

    promesas_hasta = promesas_factura[promesas_factura['fecha_promesa'] <= fecha_corte].sort_values('fecha_promesa')
    if len(promesas_hasta) == 0:
        return 0

    ultima_promesa = promesas_hasta.iloc[-1]
    return int((fecha_corte < ultima_promesa['fecha_compromiso']) and (fecha_pago_real > fecha_corte))


def promesa_resuelta_al_corte(row: pd.Series, fecha_corte: pd.Timestamp) -> bool:
    """Indica si el resultado de una promesa ya era observable al corte."""
    fecha_pago_real = row['fecha_pago_real']
    fecha_compromiso = row['fecha_compromiso']
    return bool((fecha_pago_real <= fecha_corte) or (fecha_compromiso < fecha_corte and fecha_pago_real > fecha_compromiso))


print('Funciones auxiliares definidas.')

Funciones auxiliares definidas.


## 6. Generación de clientes

Se generan `N_CLIENTES` empresas con sus atributos base.
El campo `perfil_pago` es interno de la simulación y **no entrará al modelo de IA**.

In [6]:
print('Generando clientes...')
clientes = []

for i in range(N_CLIENTES):
    perfil     = elegir_perfil()
    antiguedad = random.randint(3, 72)

    # Los clientes de mejor perfil tienen más probabilidad de tener garantía
    tiene_garantia = random.random() < (
        0.70 if perfil == 'excelente' else
        0.40 if perfil == 'regular'   else
        0.20 if perfil == 'riesgoso'  else
        0.05
    )

    clientes.append({
        'cliente_id':       f'CLI{i+1:04d}',
        'nombre':           fake.company(),
        'sector':           random.choice(SECTORES),
        'antiguedad_meses': antiguedad,
        'tiene_garantia':   int(tiene_garantia),
        'perfil_pago':      perfil,
    })

df_clientes = pd.DataFrame(clientes)

print(f'  → {len(df_clientes)} clientes generados')
print()
print('Distribución de perfiles:')
print(df_clientes['perfil_pago'].value_counts().to_string())
df_clientes.head(3)

Generando clientes...
  → 200 clientes generados

Distribución de perfiles:
perfil_pago
regular      78
excelente    56
riesgoso     47
critico      19


,cliente_id,nombre,sector,antiguedad_meses,tiene_garantia,perfil_pago
0,CLI0001,Alvarado-Laureano,agro,17,1,regular
1,CLI0002,Grupo Chávez y Canales,manufactura,34,0,critico
2,CLI0003,Villanueva-Pichardo,salud,72,1,riesgoso


## 7. Generación de facturas

Cada cliente genera entre 4 y 48 facturas distribuidas en el período 2023–2024.
La clase de mora de cada factura depende del perfil del cliente y de cuántas
moras consecutivas lleva acumuladas (efecto de deterioro).

In [7]:
print('Generando facturas con historial temporal...')
facturas_brutas = []

for _, cliente in df_clientes.iterrows():
    n_facturas         = random.randint(4, 48)
    fechas_emision     = fechas_en_rango(FECHA_INICIO, FECHA_FIN, n_facturas)
    moras_consecutivas = 0

    for fecha_emision in fechas_emision:
        condicion_dias    = random.choice(CONDICIONES_CREDITO)
        fecha_vencimiento = fecha_emision + timedelta(days=condicion_dias)
        clase_mora        = simular_clase_mora(cliente['perfil_pago'], moras_consecutivas)
        dias_mora         = dias_mora_reales(clase_mora)
        fecha_pago        = fecha_vencimiento + timedelta(days=dias_mora)

        if clase_mora == 'on_time':
            moras_consecutivas = max(0, moras_consecutivas - 1)
        else:
            moras_consecutivas += 1

        facturas_brutas.append({
            'cliente_id':        cliente['cliente_id'],
            'fecha_emision':     pd.to_datetime(fecha_emision.date()),
            'fecha_vencimiento': pd.to_datetime(fecha_vencimiento.date()),
            'fecha_pago_real':   pd.to_datetime(fecha_pago.date()),
            'condicion_dias':    condicion_dias,
            'monto':             monto_factura(cliente['sector']),
            'target_mora':       clase_mora,
            'dias_mora_real':    dias_mora,
        })

df_facturas = pd.DataFrame(facturas_brutas).sort_values(
    ['fecha_emision', 'cliente_id', 'fecha_vencimiento', 'fecha_pago_real']
).reset_index(drop=True)

# Reasignar factura_id en orden cronológico global
df_facturas.insert(0, 'factura_id', [f'FAC{i+1:06d}' for i in range(len(df_facturas))])

print(f'  → {len(df_facturas)} facturas generadas')
print()
print('Distribución del target (target_mora):')
dist = df_facturas['target_mora'].value_counts()
for cls in CLASES_MORA:
    n   = dist.get(cls, 0)
    pct = 100 * n / len(df_facturas)
    print(f'  {cls:<10} {n:>5} ({pct:.1f}%)')
df_facturas.head(3)

Generando facturas con historial temporal...


  → 5338 facturas generadas

Distribución del target (target_mora):
  on_time     2217 (41.5%)
  +30         1246 (23.3%)
  +60         1088 (20.4%)
  +90          787 (14.7%)


,factura_id,cliente_id,fecha_emision,fecha_vencimiento,fecha_pago_real,condicion_dias,monto,target_mora,dias_mora_real
0,FAC000001,CLI0051,2023-01-01,2023-04-01,2023-07-27,90,8050.57,+90,117
1,FAC000002,CLI0054,2023-01-01,2023-03-02,2023-03-29,60,5952.48,+30,27
2,FAC000003,CLI0086,2023-01-01,2023-03-02,2023-03-07,60,3613.81,+30,5


## 8. Generación de gestiones de cobranza

Solo las facturas en mora reciben gestiones.
Las gestiones escalan de canal en canal (WhatsApp → email → llamada → visita → carta notarial).
Las fechas de cada gestión se distribuyen entre el vencimiento y el pago real.

In [8]:
print('Generando gestiones de cobranza...')

perfil_cliente     = df_clientes.set_index('cliente_id')['perfil_pago'].to_dict()
gestiones          = []
gestion_id_counter = 1
gestiones_con_promesa = []

for _, fac in df_facturas.iterrows():
    target = fac['target_mora']
    perfil = perfil_cliente[fac['cliente_id']]

    lo, hi = GESTIONES_POR_CLASE[target]
    n_total = random.randint(lo, hi)
    if n_total == 0:
        continue

    # Cantidad de gestiones preventivas
    if target == 'on_time':
        n_prev = n_total
    else:
        n_prev = 0
        if n_total >= 2 and random.random() < 0.55:
            n_prev += 1
        if n_total >= 4 and fac['condicion_dias'] >= 45 and random.random() < 0.20:
            n_prev += 1
        n_prev = min(n_prev, max(n_total - 1, 0))

    n_react = n_total - n_prev

    fechas_prev = []
    inicio_prev = fac['fecha_emision'] + timedelta(days=3)
    fin_prev    = fac['fecha_vencimiento'] - timedelta(days=1)
    if n_prev > 0 and inicio_prev <= fin_prev:
        fechas_prev = muestrear_fechas_unicas(inicio_prev, fin_prev, n_prev)

    fechas_react = []
    inicio_react = fac['fecha_vencimiento'] + timedelta(days=1)
    fin_react    = fac['fecha_pago_real']
    if n_react > 0 and inicio_react <= fin_react:
        fechas_react = muestrear_fechas_unicas(inicio_react, fin_react, n_react)

    fechas_gest = [(f, 'preventiva') for f in fechas_prev] + [(f, 'reactiva') for f in fechas_react]
    fechas_gest = sorted(fechas_gest, key=lambda x: x[0])

    for intento, (fecha_g, etapa) in enumerate(fechas_gest):
        canal = canal_para_intento(intento)

        # Antes del vencimiento no se escala a visita/carta notarial
        if etapa == 'preventiva' and canal in ('visita', 'carta_notarial'):
            canal = 'llamada'

        prob_contacto = PROB_CONTACTO[canal]
        if perfil == 'critico':
            prob_contacto *= 0.70
        elif perfil == 'riesgoso':
            prob_contacto *= 0.85

        if etapa == 'preventiva' and canal == 'email':
            prob_contacto *= 0.95

        contacto_exitoso = int(random.random() < prob_contacto)

        if contacto_exitoso:
            resultado = simular_resultado_gestion(etapa, target, perfil, canal, intento)
        else:
            resultado = resultado_no_contacto(canal)

        motivo = None
        if (
            contacto_exitoso == 1 and
            fecha_g > fac['fecha_vencimiento'] and
            resultado not in ('pagado', 'confirma_pago')
        ):
            motivo = random.choice(MOTIVOS_NO_PAGO)

        dias_mora_ahora = max(0, (fecha_g - fac['fecha_vencimiento']).days)

        gestion_id = f'GES{gestion_id_counter:06d}'
        gestiones.append({
            'gestion_id':           gestion_id,
            'factura_id':           fac['factura_id'],
            'cliente_id':           fac['cliente_id'],
            'fecha_gestion':        fecha_g,
            'canal':                canal,
            'contacto_exitoso':     contacto_exitoso,
            'resultado':            resultado,
            'motivo_no_pago':       motivo,
            'dias_mora_en_gestion': dias_mora_ahora,
        })

        if resultado == 'promesa_de_pago':
            gestiones_con_promesa.append({
                'gestion_id':       gestion_id,
                'factura_id':       fac['factura_id'],
                'cliente_id':       fac['cliente_id'],
                'fecha_gestion':    fecha_g,
                'fecha_vencimiento': fac['fecha_vencimiento'],
                'fecha_pago_real':  fac['fecha_pago_real'],
                'perfil':           perfil,
            })

        gestion_id_counter += 1

df_gestiones = pd.DataFrame(gestiones)
if len(df_gestiones) > 0:
    df_gestiones['fecha_gestion'] = pd.to_datetime(df_gestiones['fecha_gestion'])

print(f'  → {len(df_gestiones)} gestiones generadas')
print(f'  → {len(gestiones_con_promesa)} gestiones terminaron en promesa de pago')
print()
print('Distribución de canales:')
print(df_gestiones['canal'].value_counts().to_string())
print()
print('Distribución de resultados:')
print(df_gestiones['resultado'].value_counts().to_string())

Generando gestiones de cobranza...


  → 14333 gestiones generadas
  → 1741 gestiones terminaron en promesa de pago

Distribución de canales:
canal
whatsapp          4221
llamada           3854
email             2704
visita            2255
carta_notarial    1299

Distribución de resultados:
resultado
no_contesta           5010
promesa_de_pago       1741
rechazo_pago          1592
en_proceso_interno    1403
disputa_monto         1294
cliente_ausente       1242
confirma_pago         1098
pagado                 509
numero_invalido        444


## 9. Generación de promesas de pago

Se crea una promesa por cada gestión que terminó en `promesa_de_pago`.

**Lógica clave de `se_cumplio`:**
- Si `fecha_compromiso <= FECHA_CORTE`: la promesa ya debería haberse resuelto → se asigna 0 o 1.
- Si `fecha_compromiso > FECHA_CORTE`: la promesa aún está pendiente al "hoy" del sistema → se asigna `null`.

Esto permite que la feature `tiene_promesa_activa` funcione correctamente en los cortes.

In [9]:
print('Generando promesas de pago...')
promesas           = []
promesa_id_counter = 1

for p in gestiones_con_promesa:
    fecha_gest       = pd.to_datetime(p['fecha_gestion'])
    fecha_pago_real  = pd.to_datetime(p['fecha_pago_real'])
    fecha_compromiso = fecha_compromiso_desde_pago(fecha_gest, fecha_pago_real, p['perfil'])
    se_cumplio       = int(fecha_pago_real <= fecha_compromiso)

    promesas.append({
        'promesa_id':       f'PRO{promesa_id_counter:06d}',
        'gestion_id':       p['gestion_id'],
        'factura_id':       p['factura_id'],
        'cliente_id':       p['cliente_id'],
        'fecha_promesa':    fecha_gest.normalize(),
        'fecha_compromiso': pd.to_datetime(fecha_compromiso).normalize(),
        'se_cumplio':       se_cumplio,
    })
    promesa_id_counter += 1

df_promesas = pd.DataFrame(promesas)
if len(df_promesas) > 0:
    df_promesas['fecha_promesa']    = pd.to_datetime(df_promesas['fecha_promesa'])
    df_promesas['fecha_compromiso'] = pd.to_datetime(df_promesas['fecha_compromiso'])

print(f'  → {len(df_promesas)} promesas generadas')
print()
cumplidas = (df_promesas['se_cumplio'] == 1).sum() if len(df_promesas) > 0 else 0
rotas     = (df_promesas['se_cumplio'] == 0).sum() if len(df_promesas) > 0 else 0
print('Estado de promesas:')
print(f'  Cumplidas:  {cumplidas}')
print(f'  Rotas:      {rotas}')
print()

if len(df_promesas) > 0:
    df_prom_cli = df_promesas.merge(df_clientes[['cliente_id','perfil_pago']], on='cliente_id')
    print('Tasa de cumplimiento por perfil:')
    for perfil in ['excelente', 'regular', 'riesgoso', 'critico']:
        sub = df_prom_cli[df_prom_cli['perfil_pago'] == perfil]
        if len(sub) > 0:
            tasa = sub['se_cumplio'].mean()
            print(f'  {perfil:<12} {tasa:.1%}  ({len(sub)} promesas)')

Generando promesas de pago...
  → 1741 promesas generadas

Estado de promesas:
  Cumplidas:  793
  Rotas:      948

Tasa de cumplimiento por perfil:
  excelente    83.2%  (131 promesas)
  regular      60.3%  (693 promesas)
  riesgoso     33.4%  (658 promesas)
  critico      17.8%  (259 promesas)


## 10. Construcción del dataset de features con cortes temporales

Esta es la parte más importante del notebook.

Por cada factura se generan múltiples filas — una por corte de scoring:
- **Corte 0** (`num_corte = 0`): Al emitir. Features de gestiones y promesas valen 0 o neutral.
- **Corte N** (`num_corte = N`): Después de la N-ésima gestión. Features acumulan lo ocurrido hasta ese momento.

Todas las filas de una misma factura comparten el mismo `target`.

**Regla anti-leakage:** En cada corte, solo se usa información disponible hasta `fecha_corte`.
Nunca se usa información futura respecto al momento del corte.

In [10]:
print('Construyendo dataset de features con cortes temporales...')
print('(Esto puede tomar 1-2 minutos)')

df_base = df_facturas.merge(
    df_clientes[['cliente_id', 'antiguedad_meses', 'tiene_garantia', 'sector']],
    on='cliente_id', how='left'
).sort_values(['cliente_id', 'fecha_emision', 'factura_id']).reset_index(drop=True)

gest_by_fact = {
    factura_id: sub.sort_values('fecha_gestion').reset_index(drop=True)
    for factura_id, sub in df_gestiones.groupby('factura_id')
} if len(df_gestiones) > 0 else {}

prom_join = df_promesas.merge(
    df_facturas[['factura_id', 'fecha_pago_real']],
    on='factura_id', how='left'
) if len(df_promesas) > 0 else pd.DataFrame()

prom_by_fact = {
    factura_id: sub.sort_values('fecha_promesa').reset_index(drop=True)
    for factura_id, sub in prom_join.groupby('factura_id')
} if len(prom_join) > 0 else {}

registros = []

for cliente_id, grupo in df_base.groupby('cliente_id'):
    grupo = grupo.sort_values(['fecha_emision', 'factura_id']).reset_index(drop=True)

    gest_cli = df_gestiones[df_gestiones['cliente_id'] == cliente_id].copy() if len(df_gestiones) > 0 else pd.DataFrame()
    prom_cli = prom_join[prom_join['cliente_id'] == cliente_id].copy() if len(prom_join) > 0 else pd.DataFrame()

    for idx, fac in grupo.iterrows():
        gest_fac = gest_by_fact.get(fac['factura_id'], pd.DataFrame())
        prom_fac = prom_by_fact.get(fac['factura_id'], pd.DataFrame())

        fechas_corte = [fac['fecha_emision']]
        if len(gest_fac) > 0:
            fechas_corte.extend(gest_fac['fecha_gestion'].tolist())

        # ── Historial previo del cliente calculado al emitir la factura ──────
        previous_pool = grupo.iloc[:idx]
        previous_ids = previous_pool['factura_id'].tolist()
        num_facturas_prev = len(previous_pool)
        fecha_base_hist = fac['fecha_emision']

        previas_resueltas = previous_pool[previous_pool['fecha_pago_real'] <= fecha_base_hist]
        if len(previas_resueltas) == 0:
            mora_prom_hist = 0.0
            tasa_cumpl = 1.0
            mora_ult_tramo = 0.0
        else:
            mora_prom_hist = previas_resueltas['dias_mora_real'].mean()
            tasa_cumpl = (previas_resueltas['target_mora'] == 'on_time').mean()
            mora_ult_tramo = previas_resueltas.tail(3)['dias_mora_real'].mean()

        monto_prom_hist = previous_pool['monto'].mean() if len(previous_pool) > 0 else fac['monto']
        ratio_monto = round(fac['monto'] / monto_prom_hist, 4) if monto_prom_hist > 0 else 1.0

        moras_cons = 0
        for _, prev in previous_pool.iloc[::-1].iterrows():
            if prev['fecha_pago_real'] > fecha_base_hist:
                break
            if prev['target_mora'] != 'on_time':
                moras_cons += 1
            else:
                break

        if len(gest_cli) > 0 and previous_ids:
            gest_hist = gest_cli[
                gest_cli['factura_id'].isin(previous_ids) &
                (gest_cli['fecha_gestion'] <= fecha_base_hist)
            ]
            tasa_contacto = round(gest_hist['contacto_exitoso'].mean(), 4) if len(gest_hist) > 0 else 0.5
        else:
            tasa_contacto = 0.5

        if len(prom_cli) > 0 and previous_ids:
            prom_prev = prom_cli[
                prom_cli['factura_id'].isin(previous_ids) &
                (prom_cli['fecha_promesa'] <= fecha_base_hist)
            ].copy()

            if len(prom_prev) > 0:
                resolved_mask = (
                    (prom_prev['fecha_pago_real'] <= fecha_base_hist) |
                    (
                        (prom_prev['fecha_compromiso'] < fecha_base_hist) &
                        (prom_prev['fecha_pago_real'] > prom_prev['fecha_compromiso'])
                    )
                )
                prom_prev_resueltas = prom_prev[resolved_mask]
                num_prom_rotas = int((prom_prev_resueltas['se_cumplio'] == 0).sum())
                total_prom = len(prom_prev)
                tasa_cumpl_prom = round(prom_prev_resueltas['se_cumplio'].mean(), 4) if len(prom_prev_resueltas) > 0 else 1.0
            else:
                num_prom_rotas = 0
                total_prom = 0
                tasa_cumpl_prom = 1.0
        else:
            num_prom_rotas = 0
            total_prom = 0
            tasa_cumpl_prom = 1.0

        # ── Variables dinámicas de la factura actual por corte ────────────────
        for num_corte, fecha_corte in enumerate(fechas_corte):
            fecha_corte = pd.to_datetime(fecha_corte)

            gest_hasta = gest_fac[gest_fac['fecha_gestion'] <= fecha_corte] if len(gest_fac) > 0 else pd.DataFrame()
            num_gest = len(gest_hasta)

            if num_gest == 0:
                dias_desde_ult = None
                ultimo_resultado_enc = None
                no_contactos_cons = 0
                disputa_activa = 0
            else:
                fechas_hist = gest_hasta['fecha_gestion'].tolist()
                if len(fechas_hist) >= 2:
                    dias_desde_ult = (fechas_hist[-1] - fechas_hist[-2]).days
                else:
                    dias_desde_ult = (fechas_hist[-1] - fac['fecha_emision']).days

                ultimo_resultado = gest_hasta.iloc[-1]['resultado']
                ultimo_resultado_enc = RESULTADO_ENC.get(ultimo_resultado, None)

                no_contactos_cons = 0
                for r in gest_hasta['resultado'].iloc[::-1].tolist():
                    if r in RESULTADOS_NO_CONTACTO:
                        no_contactos_cons += 1
                    else:
                        break

                gest_contacto = gest_hasta[gest_hasta['contacto_exitoso'] == 1]
                disputa_activa = int(len(gest_contacto) > 0 and gest_contacto.iloc[-1]['resultado'] == 'disputa_monto')

            tiene_prom_activa = promesa_activa_en_corte(prom_fac, fecha_corte, fac['fecha_pago_real']) if len(prom_fac) > 0 else 0

            dias_desde_emision = (fecha_corte - fac['fecha_emision']).days
            dias_hasta_vence = (fac['fecha_vencimiento'] - fecha_corte).days

            registros.append({
                'factura_id':                fac['factura_id'],
                'cliente_id':                cliente_id,
                'num_corte':                 num_corte,
                'fecha_corte':               fecha_corte.normalize(),
                'monto':                     round(fac['monto'], 2),
                'condicion_dias':            fac['condicion_dias'],
                'antiguedad_meses':          fac['antiguedad_meses'],
                'tiene_garantia':            fac['tiene_garantia'],
                **{f'sector_{s}': int(fac['sector'] == s) for s in SECTORES},
                'num_facturas_prev':         num_facturas_prev,
                'mora_promedio_hist':        round(mora_prom_hist, 2),
                'mora_ultimo_tramo':         round(mora_ult_tramo, 2),
                'tasa_cumplimiento':         round(tasa_cumpl, 4),
                'monto_promedio_hist':       round(monto_prom_hist, 2),
                'ratio_monto':               ratio_monto,
                'moras_consecutivas':        moras_cons,
                'num_gestiones_factura':     num_gest,
                'dias_desde_ultima_gestion': dias_desde_ult,
                'dias_desde_emision':        dias_desde_emision,
                'dias_hasta_vence':          dias_hasta_vence,
                'tasa_contacto_cliente':     tasa_contacto,
                'ultimo_resultado_enc':      ultimo_resultado_enc,
                'num_no_contesta_cons':      no_contactos_cons,
                'tiene_disputa_activa':      disputa_activa,
                'tiene_promesa_activa':      tiene_prom_activa,
                'num_promesas_rotas':        num_prom_rotas,
                'tasa_cumpl_promesas':       tasa_cumpl_prom,
                'promesas_total':            total_prom,
                'target':                    fac['target_mora'],
            })

df_features = pd.DataFrame(registros)

print(f'  → {len(df_features)} filas totales en features_ml.csv')
print(f'  → Promedio de cortes por factura: {len(df_features)/len(df_facturas):.1f}')
print(f'  → Columnas totales: {len(df_features.columns)}')

Construyendo dataset de features con cortes temporales...
(Esto puede tomar 1-2 minutos)


  → 19671 filas totales en features_ml.csv
  → Promedio de cortes por factura: 3.7
  → Columnas totales: 36


## 11. Validaciones del dataset

In [11]:
print('=== Validaciones del dataset ===')

# 1) Unicidad de IDs
assert df_clientes['cliente_id'].is_unique, 'cliente_id duplicado'
assert df_facturas['factura_id'].is_unique, 'factura_id duplicado'
assert df_gestiones['gestion_id'].is_unique, 'gestion_id duplicado'
assert df_promesas['promesa_id'].is_unique, 'promesa_id duplicado'

# 2) Facturas en orden cronológico global por factura_id
df_facturas_ord = df_facturas.sort_values('factura_id')
assert df_facturas_ord['fecha_emision'].is_monotonic_increasing, 'factura_id no respeta orden cronológico'

# 3) Coherencia de fechas en facturas
assert (df_facturas['fecha_vencimiento'] >= df_facturas['fecha_emision']).all(), 'fecha_vencimiento < fecha_emision'
assert (df_facturas['fecha_pago_real'] >= df_facturas['fecha_vencimiento']).all(), 'fecha_pago_real < fecha_vencimiento'

# 4) target_mora coherente con dias_mora_real
target_esperado = np.select(
    [
        df_facturas['dias_mora_real'] == 0,
        df_facturas['dias_mora_real'].between(1, 30),
        df_facturas['dias_mora_real'].between(31, 60),
        df_facturas['dias_mora_real'] >= 61,
    ],
    ['on_time', '+30', '+60', '+90'],
    default='on_time'
)
assert (target_esperado == df_facturas['target_mora']).all(), 'target_mora inconsistente con dias_mora_real'

# 5) Gestiones dentro de la vida útil de la factura
gest_chk = df_gestiones.merge(
    df_facturas[['factura_id', 'fecha_emision', 'fecha_pago_real', 'fecha_vencimiento']],
    on='factura_id', how='left'
)
assert (gest_chk['fecha_gestion'] >= gest_chk['fecha_emision']).all(), 'gestión antes de emisión'
assert (gest_chk['fecha_gestion'] <= gest_chk['fecha_pago_real']).all(), 'gestión después del pago real'
assert (gest_chk['dias_mora_en_gestion'] == np.maximum(0, (gest_chk['fecha_gestion'] - gest_chk['fecha_vencimiento']).dt.days)).all(), 'dias_mora_en_gestion inconsistente'

# 6) Coherencia contacto_exitoso ↔ resultado
assert set(df_gestiones.loc[df_gestiones['contacto_exitoso'] == 0, 'resultado']).issubset(RESULTADOS_NO_CONTACTO), 'resultado inválido para no contacto'
assert set(df_gestiones.loc[df_gestiones['contacto_exitoso'] == 1, 'resultado']).issubset(RESULTADOS_CONTACTO), 'resultado inválido para contacto exitoso'

# 7) Coherencia canal ↔ resultado
mask_invalido = df_gestiones['resultado'] == 'numero_invalido'
assert (~df_gestiones.loc[mask_invalido, 'canal'].isin(['visita', 'carta_notarial', 'email'])).all(), 'numero_invalido en canal incoherente'
mask_ausente = df_gestiones['resultado'] == 'cliente_ausente'
assert (~df_gestiones.loc[mask_ausente, 'canal'].isin(['whatsapp', 'email'])).all(), 'cliente_ausente en canal incoherente'

# 8) motivo_no_pago solo cuando corresponde
mask_motivo = df_gestiones['motivo_no_pago'].notna()
assert (df_gestiones.loc[mask_motivo, 'contacto_exitoso'] == 1).all(), 'motivo_no_pago con contacto fallido'
gest_motivo_chk = df_gestiones.loc[mask_motivo].merge(
    df_facturas[['factura_id', 'fecha_vencimiento']],
    on='factura_id', how='left'
)
assert (gest_motivo_chk['fecha_gestion'] > gest_motivo_chk['fecha_vencimiento']).all(), 'motivo_no_pago antes del vencimiento'
assert (~gest_motivo_chk['resultado'].isin(['pagado', 'confirma_pago'])).all(), 'motivo_no_pago con resultado de pago'

# 9) Coherencia de promesas
if len(df_promesas) > 0:
    prom_chk = df_promesas.merge(
        df_gestiones[['gestion_id', 'fecha_gestion', 'resultado']],
        on='gestion_id', how='left'
    ).merge(
        df_facturas[['factura_id', 'fecha_pago_real']],
        on='factura_id', how='left'
    )
    assert (prom_chk['resultado'] == 'promesa_de_pago').all(), 'promesa sin gestión origen válida'
    assert (prom_chk['fecha_promesa'] == prom_chk['fecha_gestion'].dt.normalize()).all(), 'fecha_promesa distinta a fecha_gestion'
    assert (prom_chk['fecha_compromiso'] >= prom_chk['fecha_promesa'] + pd.Timedelta(days=1)).all(), 'fecha_compromiso inválida'
    assert ((prom_chk['se_cumplio'] == 1) == (prom_chk['fecha_pago_real'] <= prom_chk['fecha_compromiso'])).all(), 'se_cumplio inconsistente'

# 10) Validación básica de promesa activa en features
if len(df_promesas) > 0:
    feat_prom = df_features[df_features['tiene_promesa_activa'] == 1][['factura_id', 'fecha_corte']]
    if len(feat_prom) > 0:
        merged = feat_prom.merge(df_promesas, on='factura_id', how='left').merge(
            df_facturas[['factura_id', 'fecha_pago_real']], on='factura_id', how='left'
        )
        cond = (
            (merged['fecha_promesa'] <= merged['fecha_corte']) &
            (merged['fecha_corte'] < merged['fecha_compromiso']) &
            (merged['fecha_pago_real'] > merged['fecha_corte'])
        )
        assert merged.groupby(['factura_id', 'fecha_corte'])['promesa_id'].apply(lambda s: len(s) > 0).all(), 'fila activa sin promesa'
        assert cond.groupby([merged['factura_id'], merged['fecha_corte']]).any().all(), 'promesa activa inconsistente'

print('Todas las validaciones pasaron correctamente.')
print()

print('=== Distribución de target por corte ===')
print(df_features.groupby('num_corte')['target'].value_counts(normalize=True).round(3).head(20))
print()

print('=== Distribución de cortes por factura ===')
cortes_x_fac = df_features.groupby('factura_id')['num_corte'].max()
print(cortes_x_fac.value_counts().sort_index().to_string())
print()

print('=== Promesas activas (tiene_promesa_activa = 1) ===')
print(f"  Total filas con promesa activa: {(df_features['tiene_promesa_activa'] == 1).sum()}")
print()

print('=== Valores nulos por columna ===')
nulos = df_features.isnull().sum()
nulos = nulos[nulos > 0]
print(nulos.to_string() if len(nulos) > 0 else '  Sin valores nulos inesperados.')
print()

print('=== Muestra: historial de una factura en mora ===')
fac_ejemplo = df_features[df_features['target'] == '+90']['factura_id'].iloc[0]
cols_muestra = [
    'factura_id', 'num_corte', 'fecha_corte', 'num_gestiones_factura',
    'ultimo_resultado_enc', 'tiene_promesa_activa', 'target'
]
print(df_features[df_features['factura_id'] == fac_ejemplo][cols_muestra].to_string(index=False))

=== Validaciones del dataset ===
Todas las validaciones pasaron correctamente.

=== Distribución de target por corte ===
num_corte  target 
0          on_time    0.415
           +30        0.233
           +60        0.204
           +90        0.147
1          +30        0.295
           on_time    0.261
           +60        0.258
           +90        0.186
2          +60        0.402
           +30        0.307
           +90        0.291
3          +60        0.480
           +90        0.347
           +30        0.173
4          +60        0.504
           +90        0.496
5          +90        0.588
           +60        0.412
6          +90        0.715
           +60        0.285
Name: proportion, dtype: float64

=== Distribución de cortes por factura ===
num_corte
0     1117
1     1517
2      437
3      680
4      248
5      423
6      395
7      127
8      136
9      132
10     126

=== Promesas activas (tiene_promesa_activa = 1) ===
  Total filas con promesa activa: 3415


## 12. Guardar archivos CSV

En Google Colab los archivos se guardan en el entorno temporal.
Descárgalos manualmente desde el panel de archivos (ícono de carpeta a la izquierda)
o usa el bloque de descarga automática al final de esta celda.

In [12]:
df_clientes.to_csv(OUTPUT_DIR / 'clientes.csv',            index=False)
df_facturas.to_csv(OUTPUT_DIR / 'facturas.csv',            index=False)
df_gestiones.to_csv(OUTPUT_DIR / 'gestiones_cobranza.csv', index=False)
df_promesas.to_csv(OUTPUT_DIR / 'promesas_pago.csv',       index=False)
df_features.to_csv(OUTPUT_DIR / 'features_ml.csv',         index=False)

print('Archivos guardados:')
print(f'  ruta de salida          â†’ {OUTPUT_DIR}')
print(f'  clientes.csv            → {len(df_clientes):>6} filas')
print(f'  facturas.csv            → {len(df_facturas):>6} filas')
print(f'  gestiones_cobranza.csv  → {len(df_gestiones):>6} filas')
print(f'  promesas_pago.csv       → {len(df_promesas):>6} filas')
print(f'  features_ml.csv         → {len(df_features):>6} filas  ← entrada al modelo')

Archivos guardados:
  ruta de salida          â†’ C:\Users\KevinUbe\Documents\Proyectos\GRUPO_3\GRUPO_3\artifacts\01_generacion
  clientes.csv            →    200 filas
  facturas.csv            →   5338 filas
  gestiones_cobranza.csv  →  14333 filas
  promesas_pago.csv       →   1741 filas
  features_ml.csv         →  19671 filas  ← entrada al modelo


In [13]:
# Descarga automática en Google Colab
# Si estás en Jupyter local, comenta este bloque
try:
    from google.colab import files
    for archivo in ['clientes.csv','facturas.csv','gestiones_cobranza.csv',
                    'promesas_pago.csv','features_ml.csv']:
        files.download(archivo)
    print('Descarga iniciada.')
except ImportError:
    print('No estás en Colab — los archivos ya están guardados en el directorio actual.')

No estás en Colab — los archivos ya están guardados en el directorio actual.


## 13. Resumen final

Estadísticas globales de todo el dataset generado.

In [14]:
print('══════════════════════════════════════════════════')
print(' RESUMEN FINAL DEL DATASET')
print('══════════════════════════════════════════════════')
print(f'  Clientes:                    {len(df_clientes):>6}')
print(f'  Facturas:                    {len(df_facturas):>6}')
print(f'  Gestiones de cobranza:       {len(df_gestiones):>6}')
print(f'  Promesas de pago:            {len(df_promesas):>6}')
print(f'  Filas en features_ml:        {len(df_features):>6}')
print(f'  Features por fila:           {len(df_features.columns) - 4:>6}  (excl. ids, fecha, target)')
print()
print('Distribución del target en features_ml:')
dist = df_features['target'].value_counts()
for cls in CLASES_MORA:
    n   = dist.get(cls, 0)
    pct = 100 * n / len(df_features)
    bar = '█' * int(pct / 2)
    print(f'  {cls:<10} {n:>6} ({pct:.1f}%)  {bar}')
print()
print('Columnas en features_ml.csv:')
skip = {'factura_id', 'cliente_id', 'fecha_corte', 'target'}
for col in df_features.columns:
    if col not in skip:
        print(f'  · {col}')

══════════════════════════════════════════════════
 RESUMEN FINAL DEL DATASET
══════════════════════════════════════════════════
  Clientes:                       200
  Facturas:                      5338
  Gestiones de cobranza:        14333
  Promesas de pago:              1741
  Filas en features_ml:         19671
  Features por fila:               32  (excl. ids, fecha, target)

Distribución del target en features_ml:
  on_time      3317 (16.9%)  ████████
  +30          3713 (18.9%)  █████████
  +60          5965 (30.3%)  ███████████████
  +90          6676 (33.9%)  ████████████████

Columnas en features_ml.csv:
  · num_corte
  · monto
  · condicion_dias
  · antiguedad_meses
  · tiene_garantia
  · sector_retail
  · sector_manufactura
  · sector_servicios
  · sector_construccion
  · sector_agro
  · sector_tecnologia
  · sector_salud
  · sector_transporte
  · num_facturas_prev
  · mora_promedio_hist
  · mora_ultimo_tramo
  · tasa_cumplimiento
  · monto_promedio_hist
  · ratio_monto
 